<a href="https://colab.research.google.com/github/Ololade117/interp_rag/blob/main/Interp_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U FlagEmbedding

In [2]:
!pip install marker-pdf

In [8]:
!sudo apt update
!sudo apt install llama.cpp



Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Get:2 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [75.3 kB]
Get:8 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [358 B]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,733 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,171 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,071 kB]
Get:14

In [34]:
import fitz  # pip install pymupdf
import re
from pathlib import Path

def clean(s):
    return re.sub(r"\s+", " ", s).strip()

def get_title(doc):
    page = doc[0]
    h = page.rect.height
    best, best_size = None, 0
    for b in page.get_text("dict")["blocks"]:
        if b["bbox"][1] > h * 0.4:
            continue
        for l in b.get("lines", []):
            if abs(l["dir"][0] - 1) > 0.01:  # skip rotated text (arXiv stamp)
                continue
            for s in l["spans"]:
                t = s["text"].strip()
                if t and not t.lower().startswith("arxiv") and s["size"] > best_size:
                    best, best_size = b, s["size"]
    if best is None:
        return (doc.metadata.get("title") or "").strip()
    lines = ["".join(s["text"] for s in l["spans"]) for l in best["lines"]]
    return clean(" ".join(lines))

HEADING = re.compile(r"^(?:[\dA-Z]{1,3}\.?\s+)?([A-Za-z ,&:\-]{3,60})$")

def find_heading(lines, pattern, start=0):
    for i in range(start, len(lines)):
        m = HEADING.match(lines[i].strip())
        if m and re.search(pattern, m.group(1), re.I):
            return i
    return None

def section(text, start_pat, end_pat):
    lines = text.split("\n")
    i = find_heading(lines, start_pat)
    if i is None:
        return ""
    j = find_heading(lines, end_pat, i + 1)
    j = len(lines) if j is None else j
    return clean(" ".join(lines[i + 1:j]))

def extract(path):
    doc = fitz.open(path)
    text = "\n".join(p.get_text() for p in doc)
    abstract = section(text, r"^abstract$", r"introduction|keywords?|background")
    results = section(
        text, r"\bresults\b",
        r"discussion|conclusions?|limitations|related work|references|acknowledg|appendix",
    )
    return [get_title(doc), abstract, results]

papers = [extract(p) for p in sorted(Path("/content/interp_papers").glob("*.pdf"))]

# Quick check
for p in papers[:3]:
    print(p[0][:80], "|", len(p[1]), "|", len(p[2]))

LANGUAGE MODEL ALIGNMENT IN MULTILINGUAL TROLLEY PROBLEMS | 1289 | 16845


In [2]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)

sentences = [
    "What is Retrieval-Augmented Generation?",
    "RAG is a technique that uses external data to improve LLM accuracy."
]

output = model.encode(
    sentences,
    return_sparse=True,
    return_dense=True,
    return_colbert_vecs=True

)

# 1. Dense Vectors (For semantic similarity)
# Shape: [num_sentences, 1024]
dense_embeddings = output['dense_vecs']

# 2. Sparse Vectors (For exact keyword matching / BM25-style search)
# Structure: Dictionary of token_id: weight
sparse_embeddings = output['lexical_weights']

# 3. Multi-Vector / ColBERT (For fine-grained phrase matching)
# Shape: List of 2D arrays [num_tokens_in_sentence, 1024]
multivector_embeddings = output['colbert_vecs']

print("Dense vector shape:", dense_embeddings.shape)
print("First sentence sparse terms:", list(sparse_embeddings[0].items())[:3])
print("First sentence multi-vector shape:", multivector_embeddings[0].shape)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Dense vector shape: (2, 1024)
First sentence sparse terms: [('4865', np.float32(0.11294769)), ('83', np.float32(0.114252076)), ('853', np.float32(0.14550853))]
First sentence multi-vector shape: (12, 1024)
